# CH03_04. DeepAgent의 To-do List 구현하기

<img src="./assets/agent_header_todo.png" width="800" style="display:block; margin-left:0;">

## Planning: TODO Lists

많은 에이전트들은 **TODO 리스트**를 핵심 네비게이션 도구로 활용하여 장시간 실행되는 복잡한 작업을 안정적으로 수행합니다. Claude Code는 [plan mode](https://www.anthropic.com/engineering/claude-code-best-practices)를 통해 작업 실행 전 구조화된 TODO 리스트를 먼저 생성하며, [Claude Code 프롬프트](https://cchistory.mariozechner.at/)에 정의된 `TodoWrite`라는 전용 도구를 사용합니다. 각 TODO 아이템은 두 가지 핵심 요소로 구성됩니다: **content**(짧고 구체적인 작업 설명)와 **status**(pending, in_progress, completed).

TODO 리스트의 핵심 과제는 컨텍스트 윈도우가 커질수록 **주의력을 유지하는 것**입니다. 평균적인 Manus 작업은 약 50회의 도구 호출을 수행하며, 이로 인해 [context rot](https://research.trychroma.com/context-rot)(컨텍스트 부패) 위험이 증가합니다. 에이전트는 긴 대화나 복잡한 작업 중에 주제에서 벗어나거나 이전 목표를 잊어버리기 쉽습니다. Manus와 같은 에이전트는 TODO 리스트를 지속적으로 **다시 작성하고 업데이트**함으로써, 컨텍스트 끝에서 자신의 목표를 효과적으로 암송(recite)하여 [작업에 집중을 유지](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus)하고 미션 드리프트를 방지합니다.

<!-- The style below reduces the gap between items in the same bulleted list. Run once per notebook -->
<style>
/* JupyterLab + classic notebook */
.jp-RenderedHTMLCommon ul, .text_cell_render ul { margin-top: .25em; margin-bottom: .35em; padding-left: 1.2em; }
.jp-RenderedHTMLCommon ul ul, .text_cell_render ul ul { margin-top: .15em; margin-bottom: .15em; padding-left: 1.0em; }
.jp-RenderedHTMLCommon li, .text_cell_render li { margin: .1em 0; }
</style>

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv


# 환경 변수 확인
print("=== LangSmith 설정 상태 ===")
print(f"LANGSMITH_API_KEY: {'설정됨' if os.getenv('LANGSMITH_API_KEY') else '미설정 ⚠️'}")
print(f"LANGSMITH_TRACING: {os.getenv('LANGSMITH_TRACING', '미설정')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT', '미설정 (default 사용)')}")
print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정 ⚠️'}")

# 프로젝트 이름 설정 (선택)
os.environ["LANGSMITH_PROJECT"] = "fc-agent-bible-appendix"

print("Tracing이 활성화되었습니다.")

## 2. 커스텀 상태(State) 정의

이전 레슨과 마찬가지로, `create_agent`에 커스텀 상태를 사용합니다.

State 객체는 워크플로우의 서로 다른 단계 간에 컨텍스트를 저장하고 전달하는 핵심 메커니즘입니다. State는 그래프의 스키마와 상태 업데이트 방법을 지정하는 **리듀서(reducer)** 함수로 구성됩니다.

DeepAgent 스키마에는 세 가지 핵심 요소가 정의됩니다: **`messages`**, **`todos`**, **`files`**.

- **`messages`**: `AgentState`에서 상속됩니다.
  - `add_messages` 리듀서가 새 메시지를 메시지 리스트 끝에 추가합니다.
- **`todos`**: `Todo` 작업의 리스트입니다. 각 작업에는 설명(`content`)과 상태(`status`: pending, in_progress, completed)가 있습니다.
  - 커스텀 리듀서가 없으므로, 업데이트 시 리스트를 덮어씁니다.
- **`files`**: 다음 레슨에서 탐구할 상태 내 가상 파일 시스템입니다.

In [ ]:
from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict
from langchain.agents import AgentState


class Todo(TypedDict):
    """복잡한 워크플로우의 진행 상황을 추적하기 위한 구조화된 작업 항목.

    Attributes:
        content: 작업에 대한 짧고 구체적인 설명
        status: 현재 상태 - pending, in_progress, 또는 completed
    """

    content: str
    status: Literal["pending", "in_progress", "completed"]


def file_reducer(left, right):
    """두 파일 딕셔너리를 병합하며, 오른쪽 값이 우선됩니다.

    에이전트 상태의 files 필드에 대한 리듀서 함수로 사용되어,
    가상 파일 시스템의 점진적 업데이트를 가능하게 합니다.

    Args:
        left: 왼쪽 딕셔너리 (기존 파일들)
        right: 오른쪽 딕셔너리 (새로운/업데이트된 파일들)

    Returns:
        오른쪽 값이 왼쪽 값을 덮어쓴 병합된 딕셔너리
    """
    if left is None:
        return right
    elif right is None:
        return left
    else:
        return {**left, **right}


class DeepAgentState(AgentState):
    """작업 추적 및 가상 파일 시스템을 포함하는 확장된 에이전트 상태.

    LangGraph의 AgentState를 상속하고 다음을 추가합니다:
    - todos: 작업 계획 및 진행 상황 추적을 위한 Todo 항목 리스트
    - files: 파일명을 콘텐츠에 매핑하는 딕셔너리로 저장된 가상 파일 시스템
    """

    todos: NotRequired[list[Todo]]
    files: Annotated[NotRequired[dict[str, str]], file_reducer]

## 3. TODO 도구 설명

위에서 설명했듯이, 장시간 실행되는 에이전트는 TODO 리스트를 사용하여 작업에 집중할 수 있습니다. 이를 위해 `write_todos`와 `read_todos` 도구를 만듭니다. 아래의 도구 설명은 LLM에게 TODO 리스트를 **언제** 사용할지, **무엇을** 포함할지, 그리고 어떻게 **읽고 업데이트**할지를 상세히 안내합니다.

리스트에 개별 작업이 포함되어 있지만, **전체 리스트를 다시 작성하는 방식**으로 업데이트한다는 점에 주목하세요. 이를 통해 LLM이 진행하면서 작업을 재고할 수 있습니다.

In [ ]:
from utils import show_prompt

WRITE_TODOS_DESCRIPTION = """복잡한 작업 흐름의 진행 상황을 추적하기 위해 구조화된 TODO 리스트를 생성하고 관리하세요.

## 언제 사용할까요?
- 여러 단계가 있거나 복잡한 작업이 필요한 경우
- 사용자가 여러 작업을 요청하거나 TODO 리스트 작성을 명시적으로 요구할 때
- 단일하고 단순한 작업에는 특별한 요구가 없으면 사용하지 마세요

## 구조
- content(내용), status(상태)를 가진 여러 개의 todo 객체를 하나의 리스트로 관리하세요
- 명확하고 실행 가능한 작업 설명을 사용하세요
- status는 반드시: pending(대기 중), in_progress(진행 중), completed(완료) 중 하나여야 합니다

## 모범 사례  
- 동시에 진행 중(in_progress)인 작업은 한 개만 유지하세요
- 작업이 완전히 끝나면 즉시 completed(완료)로 표시하세요
- 변경 시마다 항상 전체 최신 리스트를 다시 보내세요
- 불필요한 항목은 정리하여 리스트를 집중되게 관리하세요

## 진행 상황 업데이트
- 작업 상태 변경이나 내용 수정을 위해 TodoWrite를 다시 호출하세요
- 실시간으로 진행 상황을 반영하세요. 여러 완료 작업을 한 번에 처리하지 마세요
- 작업이 막힌 경우, in_progress 상태를 유지하고, 새로 막힌 점을 설명하는 작업을 추가하세요

## 입력 파라미터
- todos: content와 status 필드를 포함한 TODO 항목 리스트

## 반환값
에이전트의 상태를 새로운 todo 리스트로 업데이트합니다.
"""

show_prompt(WRITE_TODOS_DESCRIPTION)

## 4. Write and Read TODO 도구 구현

아래에서 **`write_todos`**와 **`read_todos`** 도구를 구현합니다:

**`write_todos`** 도구는 LLM이 생성한 TODO 리스트를 인자로 받아 상태에 저장하고, 이전 리스트를 덮어씁니다. 그런 다음 작성된 리스트가 포함된 `ToolMessage`를 반환합니다. 리스트를 작성하면 LLM이 생성한 도구 호출과 반환된 `ToolMessage` 양쪽 모두에서 `messages`에 저장된 대화 히스토리를 통해 LLM이 해당 정보에 접근할 수 있게 됩니다.

**`read_todos`** 도구는 상태에서 TODO 리스트를 읽어 `ToolMessage`로 반환합니다. LLM 컨텍스트에서 정보를 새로고침하는 데 사용할 수 있습니다.

이 도구들은 이전 레슨에서 다룬 기능들을 활용합니다:
- `InjectedState`: 도구가 그래프 상태에 접근할 수 있도록 합니다.
- `Command`: 상태의 값을 업데이트합니다.

In [ ]:
from typing import Annotated

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command


@tool(description=WRITE_TODOS_DESCRIPTION, parse_docstring=True)
def write_todos(
    todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """작업 계획 및 추적을 위해 에이전트의 TODO 리스트를 생성하거나 업데이트합니다.

    Args:
        todos: content와 status를 가진 Todo 항목 리스트
        tool_call_id: 메시지 응답을 위한 도구 호출 식별자

    Returns:
        새 TODO 리스트로 에이전트 상태를 업데이트하는 Command
    """
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"Updated todo list to {todos}", tool_call_id=tool_call_id)
            ],
        }
    )


@tool(parse_docstring=True)
def read_todos(
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> str:
    """에이전트 상태에서 현재 TODO 리스트를 읽어옵니다.

    이 도구를 통해 에이전트가 현재 TODO 리스트를 검색하고 검토하여
    남은 작업에 집중하고 복잡한 워크플로우의 진행 상황을 추적할 수 있습니다.

    Args:
        state: 현재 TODO 리스트를 포함하는 주입된 에이전트 상태
        tool_call_id: 메시지 추적을 위한 주입된 도구 호출 식별자

    Returns:
        현재 TODO 리스트의 포맷팅된 문자열 표현
    """
    todos = state.get("todos", [])
    if not todos:
        return "현재 리스트에 TODO가 없습니다."

    result = "현재 TODO 리스트:\n"
    for i, todo in enumerate(todos, 1):
        status_emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result += f"{i}. {emoji} {todo['content']} ({todo['status']})\n"

    return result.strip()

## 5. 에이전트 생성

첫 번째 레슨과 마찬가지로, `create_agent`를 사용하여 에이전트를 구성합니다. 이번에는 에이전트가 TODO 리스트를 활용하도록 집중합니다. Manus의 접근 방식을 따라 각 작업 완료 후 TODO를 암송(recitation)하도록 하고, 위에서 만든 도구들을 사용합니다.

In [ ]:
TODO_USAGE_INSTRUCTIONS = """사용자의 요청에 따라 다음 지침을 따르세요:
1. 도구 설명에 따라 사용자 요청 시작 시 `write_todos` 도구를 사용하여 TODO 리스트를 생성하세요.
2. TODO를 완료한 후에는 `read_todos`를 사용하여 TODO 목록을 확인하고 계획을 상기하세요.
3. 수행한 작업과 TODO 내용을 대조하며 검토하세요.
4. 완료된 작업은 완료로 표시하고 다음 TODO로 진행하세요.
5. 모든 TODO를 완료할 때까지 이 과정을 반복하세요.

중요: 어떤 사용자 요청이든 항상 TODO 연구 계획을 수립하고 위 지침에 따라 연구를 수행해야 합니다.
중요: 관리해야 할 TODO의 수를 최소화하기 위해 연구 작업들을 가급적 *하나의 TODO*로 묶어서 처리하세요."""

show_prompt(TODO_USAGE_INSTRUCTIONS)

In [ ]:
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from utils import format_messages

# 모의 검색 결과
search_result = """Model Context Protocol(MCP)는 Anthropic에서 개발한 오픈 스탠다드 프로토콜로, 
AI 모델과 툴, 데이터베이스, 기타 외부 시스템 간의 원활한 통합을 가능하게 합니다. 이는 표준화된 통신 레이어 역할을 하여
AI 모델이 다양한 소스로부터 데이터를 일관적이고 효율적으로 접근 및 활용할 수 있도록 합니다.
즉, MCP는 AI 어시스턴트를 외부 서비스에 연결하는 과정을 단순화하며, 데이터 교환을 위한 통합 언어를 제공합니다.
"""

# 모의 검색 도구
@tool(parse_docstring=True)
def web_search(
    query: str,
):
    """특정 주제에 대한 정보를 웹에서 검색합니다.

    이 도구는 웹 검색을 수행하고 주어진 쿼리에 대한
    관련 결과를 반환합니다. 어떤 주제에 대해 인터넷에서
    정보를 수집해야 할 때 사용하세요.

    Args:
        query: 검색 쿼리 문자열. 찾고자 하는 정보에 대해
               구체적이고 명확하게 작성하세요.

    Returns:
        검색 엔진의 검색 결과.

    Example:
        web_search("헬스케어에서의 머신러닝 응용")
    """
    return search_result

# create_agent를 사용하여 에이전트 생성
model = init_chat_model(model="openai:gpt-4.1-mini", temperature=0.0)
tools = [write_todos, web_search, read_todos]

# 모의 리서치 지침 한글화
SIMPLE_RESEARCH_INSTRUCTIONS = """중요: 반드시 web_search 도구를 한 번만 호출하고, 해당 도구에서 제공된 결과를 이용해 사용자의 질문에 답변하세요."""

# 에이전트 생성
agent = create_agent(
    model,
    tools,
    system_prompt=TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + SIMPLE_RESEARCH_INSTRUCTIONS,
    state_schema=DeepAgentState,
)

# 에이전트 표시
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

## 6. 에이전트 실행

상태에 `todos`가 없는 상태로 그래프를 시작하고, 사용자 리서치 요청을 전달합니다.

In [ ]:
# 사용 예시
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Model Context Protocol (MCP)에 대한 짧은 요약을 해주세요.",
            }
        ],
        "todos": [],
    }
)

format_messages(result["messages"])